<a href="https://colab.research.google.com/github/tuankhoin/CO3133-Deep-Learning/blob/main/Week_4_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ho Chi Minh University of Technology (HCMUT)

CO3133 - Deep Learning and Its Applications

# Week 4 - Recurrent Neural Networks

### Sequences, Hidden States, and Sequence Models

This lecture introduces recurrent neural networks for sequential data:

- sequential-data motivation;
- RNN architecture and task shapes;
- vanilla RNN equations;
- backpropagation through time (BPTT);
- vanishing and exploding gradients;
- LSTM and GRU;
- practical implementation in PyTorch and Keras.

## Learning Outcomes

By the end of this lecture, you should be able to:

- Explain why sequential data needs special treatment.
- Describe the hidden-state idea in recurrent neural networks.
- Distinguish common sequence-to-output patterns.
- Write the update equations for a vanilla RNN.
- Explain BPTT and the vanishing/exploding gradient problem.
- Explain how LSTM and GRU gates improve long-range memory.
- Interpret common input/output tensor shapes used by deep-learning frameworks.

## Lecture Contents

1. Motivation: Sequential Data
2. RNN Architecture
3. Mathematical View
4. Training: BPTT and Long-Range Challenges
5. LSTM and GRU
6. Programming View
7. Summary

---
# 1. Motivation: Sequential Data
---

## Why Sequential Data Is Different

<img src='https://media.licdn.com/dms/image/v2/D4D22AQFYCWlA2hqTIg/feedshare-shrink_800/B4DZpLHDxHGgAg-/0/1762196733904?e=2147483647&v=beta&t=m288VxIBdbND4odFxYajdQl6xAMnBHCp40OS3V5c6s4' height=300 />

Examples:

- **Natural language:** tokens have order.
- **Speech and audio:** signals evolve over time.
- **Time series:** sensors, finance, IoT signals.
- **Video:** frames form a temporal sequence.

Main difficulties:

- sequence length may vary;
- positions are **not exchangeable**;
- later observations may depend on much earlier context.

## Why Not a Plain Feedforward Network?

<img src="https://24.media.tumblr.com/332e9db762ab2df743116cf07cd4d3cd/tumblr_msppdwemdN1sezoa7o1_400.gif" height=200/>

Simple feedforward approaches have limitations:

- Fixed-length vectorization can discard order.
- Treating each time step independently ignores temporal dependencies.
- Different weights for every time step are parameter-inefficient.
- Fixed-length designs may not generalize to new sequence lengths.

### What we want

A model that:

1. stores a summary of the past in a **hidden state**;
2. reuses the **same transition function at every time step**.

This is parameter sharing across time.

## Sequence Notation

Input sequence:

$$
\mathbf{x}_1,\mathbf{x}_2,\ldots,\mathbf{x}_T,
\qquad
\mathbf{x}_t\in\mathbb{R}^{D}
$$

Hidden state:

$$
\mathbf{h}_t\in\mathbb{R}^{H}
$$

Common batched shape:

$$
\mathbf{X}\in\mathbb{R}^{B\times T\times D}
$$

where:

- $B$ = batch size;
- $T$ = time steps;
- $D$ = features per step;
- $H$ = hidden size.

---
# 2. RNN Architecture
---

<img src="https://media.geeksforgeeks.org/wp-content/uploads/20250523171309383561/recurrent_neural_network.webp" height=300/>

## The RNN Cell

At time $t$, the RNN receives:

- current input $\mathbf{x}_t$;
- previous hidden state $\mathbf{h}_{t-1}$.

and produces:

$$
(\mathbf{x}_t,\mathbf{h}_{t-1})
\longrightarrow
\mathbf{h}_t.
$$

The **same parameters are reused at every time step**.

$\mathbf{h}_t$ is a compressed memory of the sequence observed up to time $t$.

## Unrolling Through Time

Conceptually:

$$
\mathbf{h}_0
\rightarrow
\mathbf{h}_1
\rightarrow
\mathbf{h}_2
\rightarrow
\cdots
\rightarrow
\mathbf{h}_T
$$

while the model receives:

$$
\mathbf{x}_1,\mathbf{x}_2,\ldots,\mathbf{x}_T.
$$

Unrolling makes dependencies explicit:

- later states depend on earlier inputs and states;
- training gradients flow backward through this chain.

This leads to **Backpropagation Through Time (BPTT)**.

## Common Sequence Task Shapes

<img src="https://goodboychan.github.io/images/copied_from_nb/image/various_rnn.png" height=300/>

| Pattern | Example | Typical readout |
|---|---|---|
| **Many-to-one** | sentiment, video classification | use $\mathbf{h}_T$ or pool hidden states |
| **Many-to-many (synchronized)** | POS tagging, frame labels | one output per input step |
| **One-to-many** | captioning, sequence generation | decode step-by-step from a seed |
| **Many-to-many (delayed)** | translation, summarization | encoder $\rightarrow$ context $\rightarrow$ decoder |

The recurrent core may stay the same; the task head and supervision points change.

## Many-to-One and Many-to-Many

### Many-to-one

One prediction for the whole sequence:

$$
\hat{\mathbf{y}}
=
\operatorname{Head}(\mathbf{h}_T)
$$

or:

$$
\hat{\mathbf{y}}
=
\operatorname{Head}
\left(
\operatorname{Pool}
(\mathbf{h}_1,\ldots,\mathbf{h}_T)
\right).
$$

### Many-to-many (synchronized)

One output per input step:

$$
\hat{\mathbf{y}}_t
=
\operatorname{Head}(\mathbf{h}_t).
$$

The same head is usually shared across time.

## One-to-Many and Encoder-Decoder

### One-to-many

A short context or seed produces a longer sequence.

During training, the next input may use the true previous token (**teacher forcing**).  
During inference, it may use the model's previous prediction.

### Many-to-many (delayed)

Input and output lengths may differ:

$$
T\neq S.
$$

An encoder produces a context representation $c$, then a decoder generates:

$$
\hat{\mathbf{y}}_1,\ldots,\hat{\mathbf{y}}_S.
$$

The context can be a final state, pooled states, or attention over encoder states.

---
# 3. Mathematical View
---

<img src="https://media.geeksforgeeks.org/wp-content/uploads/20250214171817652462/tanh.png" height=200/>

## Vanilla RNN Update

For one layer:

$$\boxed{
\mathbf{h}_t
=
\tanh
\left(
\mathbf{W}_{xh}\mathbf{x}_t
+
\mathbf{W}_{hh}\mathbf{h}_{t-1}
+
\mathbf{b}_h
\right)}
$$

with initial state often:

$$
\mathbf{h}_0=\mathbf{0}.
$$

Parameter shapes:

$$
\mathbf{W}_{xh}\in\mathbb{R}^{H\times D},
\qquad
\mathbf{W}_{hh}\in\mathbb{R}^{H\times H},
\qquad
\mathbf{b}_h\in\mathbb{R}^{H}.
$$

## Output Readout

A task-specific output can be computed as:

$$\boxed{
\mathbf{z}_t
=
\mathbf{W}_{hy}\mathbf{h}_t+\mathbf{b}_y}
$$

followed by:
$
\hat{\mathbf{y}}_t
=
\operatorname{Activation}(\mathbf{z}_t).
$

Examples:

- softmax - multiclass classification;
- sigmoid - binary / multi-label classification;
- linear output - regression.

This is just like any basic perceptron maths: `activation(ax+b)`

## Parameter Count

Ignoring the task output head:

$$\boxed{
\#\text{params}
=
HD+H^2+H}
$$

Reason:

- $\mathbf{W}_{xh}$: $HD$
- $\mathbf{W}_{hh}$: $H^2$
- $\mathbf{b}_h$: $H$

The parameter count does **not** increase with sequence length because weights are shared across time.

## Bidirectional RNN

<img src="https://images.cartoonstock.com/lowres/health-beauty-weighing_scale-future-validity-uncertainty-body_weight-CS596476_low.jpg" height=300/>

A bidirectional RNN processes the sequence in both directions:

$$
\overrightarrow{\mathbf{h}}_t
\qquad\text{and}\qquad
\overleftarrow{\mathbf{h}}_t.
$$

They are commonly concatenated:

$$
\mathbf{h}_t
=
[
\overrightarrow{\mathbf{h}}_t;
\overleftarrow{\mathbf{h}}_t
].
$$

Useful when future context is available, such as:

- sequence labeling;
- document understanding;
- offline speech processing.

Not appropriate for strictly causal prediction.

---
# 4. Training: BPTT and Long-Range Challenges
---

## Backpropagation Through Time (BPTT)

<img src="https://miro.medium.com/0*SWHzEFzYRDSnc3w2" height=300/>

Training:

1. unroll the RNN for $T$ steps;
2. compute the loss;
3. backpropagate through the temporal computation graph.

For per-step supervision:

$$
\mathcal{L}
=
\sum_{t=1}^{T}
\ell_t
(
\hat{\mathbf{y}}_t,
\mathbf{y}_t
).
$$

BPTT is ordinary backpropagation applied to the **unrolled recurrent network**.

## Why Long Sequences Are Difficult

Gradients may contain **products** such as:

$$
\frac{\partial \mathcal{L}}
{\partial \mathbf{h}_k}
=
\left(
\prod_{t=k+1}^{T}
\frac{\partial \mathbf{h}_t}
{\partial \mathbf{h}_{t-1}}
\right)^{\top}
\frac{\partial \mathcal{L}}
{\partial \mathbf{h}_T}.
$$

Repeated multiplication of Jacobians can cause:

- **vanishing gradients**: gradient shrinks toward zero;
- **exploding gradients**: gradient grows excessively large.

The effective computational depth grows with sequence length.

## Vanishing and Exploding Gradients


<img src="https://i.ytimg.com/vi/8z3DFk4VxRo/maxresdefault.jpg" height=300/>

### Vanishing gradients

- early time steps receive very weak learning signals;
- long-range dependencies become difficult to learn.

### Exploding gradients

- updates can become unstable;
- numerical problems may occur.

### Common responses

- gradient clipping for exploding gradients;
- LSTM / GRU for improved long-term information flow.

## Truncated BPTT and Practical Tips

### Truncated BPTT

<img src="https://preview.redd.it/no-model-currently-can-explain-what-the-image-means-v0-08n5u6409ydf1.jpeg?auto=webp&s=52edcff51aa29c51ea460ff6a1883e0237e0eca9" height=200/>

Backpropagate only through a limited time window.

Benefits:

- lower memory;
- lower computation;
- often more stable training.

Trade-off:

- less direct gradient information for very long dependencies.

### Practical tips

- tune learning rate carefully;
- initialize recurrent weights carefully;
- clip gradients where needed;
- mask padded positions;
- use normalization variants where appropriate.

---
# 5. LSTM and GRU
---

<img src="https://raw.githubusercontent.com/tuankhoin/CO3057-Computer-Vision/refs/heads/main/assets/claude.jpg" height=300/>

Vanilla RNNs struggle with long-range dependencies.

LSTM and GRU introduce **gates** that control how information is preserved or replaced.

## LSTM: Main Idea

3 steps: remove, add, use

<img src="https://assets.insightmediagroup.io/media/wp-content/uploads/2022/02/17cMfenu76BZCzdKWCfBABA.png" height=400/>

<img src="https://miro.medium.com/0*NIFLmgpE4TDRBwh4.gif" height=200/>

LSTM maintains 2 states:

- hidden state $\mathbf{h}_t$;
- cell state $\mathbf{c}_t$.

Main components:

| Component | Purpose |
|---|---|
| Forget gate $\mathbf{f}_t$ | decide what old information to **discard** (F=0: forget everything, F=1: remember everything) |
| Input gate $\mathbf{i}_t$ | decide what new information to **add** |
| Output gate $\mathbf{o}_t$ | decide what information to use **now** |
| Candidate $\mathbf{g}_t$ | candidate new cell content |

## LSTM Gates

Let:

$$\boxed{
\mathbf{z}_t
=
[
\mathbf{h}_{t-1};
\mathbf{x}_t
]
\in\mathbb{R}^{H+D}}
$$

Then:

$$\boxed{
\mathbf{f}_t
=
\sigma(\mathbf{W}_f\mathbf{z}_t+\mathbf{b}_f)\\
\mathbf{i}_t
=
\sigma(\mathbf{W}_i\mathbf{z}_t+\mathbf{b}_i)\\
\mathbf{o}_t
=
\sigma(\mathbf{W}_o\mathbf{z}_t+\mathbf{b}_o)\\
\mathbf{g}_t
=
\tanh(\mathbf{W}_g\mathbf{z}_t+\mathbf{b}_g)}
$$

Remember: $\sigma$ is Sigmoid function (range 0-1)

## LSTM State Update

Cell state:

$$\boxed{
\mathbf{c}_t
=
\mathbf{f}_t\odot\mathbf{c}_{t-1}
+
\mathbf{i}_t\odot\mathbf{g}_t}
$$

Hidden state:

$$\boxed{
\mathbf{h}_t
=
\mathbf{o}_t\odot\tanh(\mathbf{c}_t)}
$$

Typical initialization:

$$\boxed{
\mathbf{h}_0=\mathbf{0},
\qquad
\mathbf{c}_0=\mathbf{0}}
$$

$\odot$ means **element-wise multiplication**.

## Why LSTM Helps Long Dependencies

The update

$$
\mathbf{c}_t
=
\mathbf{f}_t\odot\mathbf{c}_{t-1}
+
\cdots
$$

contains an additive memory path.

If:

$$
\mathbf{f}_t\approx 1,
$$

old information can persist over many steps without repeatedly passing through the same tanh transformation.

The forget gate also allows **adaptive forgetting**.

## GRU: Main Idea

<img src="https://assets.insightmediagroup.io/media/wp-content/uploads/2022/02/1LfH52lSd1mq-UrWuejzO8g.png" height=400/>

<img src="https://i.imgur.com/5w37uXz.gif" height=200/>

GRU simplifies LSTM.

It uses:

- 2 gates:
  - update gate $\mathbf{z}_t$: How much to **discard and add** to the memory (permanent effect)
  - reset gate $\mathbf{r}_t$: How much information **to not use** at the moment (temporary effect)
- 1 state only: candidate hidden state $\tilde{\mathbf{h}}_t$.

### GRU vs. LSTM
GRU has **no separate cell state**.

Typical consequences:

- **fewer parameters** than LSTM;
- **simpler** hidden-state structure;
- often **faster** computation.


## GRU Equations

Let:

$$
\boxed{\boldsymbol{\xi}_t
=
[
\mathbf{h}_{t-1};
\mathbf{x}_t
]}
$$

Gates:

$$\boxed{
\mathbf{r}_t
=
\sigma(\mathbf{W}_r\boldsymbol{\xi}_t+\mathbf{b}_r)\\
\mathbf{z}_t
=
\sigma(\mathbf{W}_z\boldsymbol{\xi}_t+\mathbf{b}_z)
}
$$

Candidate:

$$\boxed{
\tilde{\mathbf{h}}_t
=
\tanh
\left(
\mathbf{W}_h
[
\mathbf{r}_t\odot\mathbf{h}_{t-1};
\mathbf{x}_t
]
+
\mathbf{b}_h
\right)}
$$

## GRU Hidden-State Update

A common form:

$$\boxed{
\mathbf{h}_t
=
(1-\mathbf{z}_t)
\odot
\tilde{\mathbf{h}}_t
+
\mathbf{z}_t
\odot
\mathbf{h}_{t-1}}
$$

Interpretation:

- $\mathbf{z}_t\approx1$ → retain old information;
- $\mathbf{z}_t\approx0$ → use the new candidate;
- $\mathbf{r}_t\approx0$ → reduce past influence when forming the candidate.

## Vanilla RNN vs LSTM vs GRU

| Model | State -- | Gates | Main Strength | Main Limitation |
|---|---|---:|---|---|
| Vanilla RNN | $\mathbf{h}_t$ | 0 | simple baseline | weak long-term memory |
| LSTM | $\mathbf{h}_t,\mathbf{c}_t$ | 3 main gates | explicit long-term memory | more parameters |
| GRU | $\mathbf{h}_t$ | 2 gates | simpler and efficient | less explicit memory separation |

---

| Feature | LSTM | GRU |
|---|---|---|
| Number of Gates | 3 (Forget, Input, Output) | 2 (Update, Reset) |
| Memory Streams | 2 (Long-term & Short-term separate) | 1 (Combined Hidden State) |
| Speed & Efficiency | Slower to train, uses more memory | Faster to train, uses less memory |
| Performance | Better for massive datasets with very complex patterns | Often matches LSTM performance on smaller datasets |

Practical guidance:

- **Vanilla RNN:** teaching and baselines.
- **LSTM / GRU:** common gated choices.
- **Bidirectional:** when full-sequence context is available.
- **Stacked:** richer representations, with extra compute and overfitting risk.

### Chạm Mặt Giang Hồ

<img src="https://cdn2.tuoitre.vn/phunuonline/image/fckeditor/upload/2020/20200421/images/phim-giang-ho-ngon-ngon-tren-_1587509665.jpg" height=150/>

<p>Mỗi tập minh họa cổng nào trong LSTM: Input, Forget hay Output?</p>

<p><b>Tập 1:</b> Nam chính: “Anh đã từng yêu gái 2k, em tin không?”</p>
<details>
  <summary>Xem đáp án</summary>
  <p><b>Input:</b> Ghi nhớ lời kể về người yêu cũ.</p>
</details>

<p><b>Tập 2:</b> Phản diện: "Gà luộc không xé thì cút! Thằng nào có bản lĩnh tới 66B Nguyễn Sỹ Sách phường 15 Tân Bình gặp tao. Tao sẵn sàng tiếp từng đứa. Tất nhiên tiền cafe mày trả.”</p>
<details>
  <summary>Xem đáp án</summary>
  <p><b>Input:</b> Ghi nhớ quy tắc ăn gà, bản lĩnh thì ra địa chỉ nào, tiền cafe ai trả...</p>
</details>

<p><b>Tập 3:</b> Khá Bảnh xuất hiện, gia nhập nhóm. Ăn gà luộc chặt.</p>
<details>
  <summary>Xem đáp án</summary>
  <p><b>1. Input:</b> Ghi nhớ nhân vật mới.</p>
  <p><b>2. Output:</b> Khá Bảnh không ăn gà luộc xé -> Khả năng là cút.</p>
</details>

<p><b>Tập 4:</b> Khá Bảnh đi tù, không còn tham gia câu chuyện.</p>
<details>
  <summary>Xem đáp án</summary>
  <p><b>Forget:</b> Bỏ Khá Bảnh khỏi bộ nhớ đang theo dõi.</p>
</details>

<p><b>Tập 10:</b> Một cô gái sinh năm 2000 xuất hiện. Người xem liên hệ cô gái với lời kể của nam chính,
nghi vấn cô ấy là người yêu cũ.</p>
<details>
  <summary>Xem đáp án</summary>
  <p><b>Input:</b> Ghi nhận nhân vật và năm sinh.</p>
  <p><b>Output:</b> Đưa ký ức đã lưu ra sử dụng để hỗ trợ suy đoán.
</details>

<p><b>Tập 14:</b> Nam chính đã đủ bản lĩnh sau nhiều tập phim.</p>
<details>
  <summary>Xem đáp án</summary>
  <p><b>Input:</b> Ghi nhận nhân vật đủ bá khí.</p>
  <p><b>Output:</b> Liệu nam chính sẽ ra 66B Nguyễn Sỹ Sách gặp phản diện?</p>
</details>

<p><b>Tập 16:</b> Tất cả cùng đi tù, hết phim.</p>
<details>
  <summary>Xem đáp án</summary>
  <p><b>Forget = 0, Input = 0:</b> Xóa ký ức cũ hoàn toàn, không ghi mới.</p>
  <p>Lưu ý: <b>Forget = 1 là giữ lại!</b></p>
</details>

---
# 6. Programming View
---

## PyTorch: `nn.RNN`

```python
rnn = nn.RNN(
    input_size=D,
    hidden_size=H,
    batch_first=True
)

out, h_n = rnn(x, h0)
```

Typical shapes:

$$
\mathbf{x}:(B,T,D)
$$

$$
\mathbf{out}:(B,T,H)
$$

$$
\mathbf{h}_n:(1,B,H)
$$

With `bidirectional=True`, output features become $2H$.

## PyTorch: LSTM and GRU

```python
lstm = nn.LSTM(
    input_size=D,
    hidden_size=H,
    num_layers=L,
    batch_first=True
)

gru = nn.GRU(
    input_size=D,
    hidden_size=H,
    num_layers=L,
    batch_first=True
)

out, (h_n, c_n) = lstm(x)
out, h_n = gru(x)
```

LSTM returns both hidden and cell states.  
GRU returns only the hidden state.

## Variable-Length Sequences

### Padding + masking

Pad sequences to a common length and prevent padded positions from affecting the loss.

### Packed sequences

PyTorch provides packed-sequence utilities to avoid unnecessary computation on padding.

Important:

> In padded batches, `out[:, -1, :]` is not necessarily the final **valid** time step for every sample.

## Many-to-One Classifier Pattern

Typical pipeline:

$$
\mathbf{X}
\rightarrow
\text{RNN/LSTM/GRU}
\rightarrow
\mathbf{h}_T
\rightarrow
\text{Linear Head}
\rightarrow
\text{Logits}
$$

Conceptually:

```python
out, _ = rnn(x)
last = out[:, -1, :]
logits = fc(last)
```

For padded batches, select the last valid state for each sequence.

## Keras View

```python
model = tf.keras.Sequential([
    layers.Input(shape=(None, D)),
    layers.LSTM(H, return_sequences=True),
    layers.LSTM(H, return_sequences=False),
    layers.Dense(K)
])
```

- `return_sequences=True` → return all time steps.
- `return_sequences=False` → return only the final step.
- Combine with masking when padding is used.

---
# 7. Summary
---

## Key Takeaways

- RNNs model sequences using a **hidden state** and **shared transition across time**.
- Hidden states summarize previous inputs.
- The same recurrent core supports many-to-one, one-to-many and many-to-many tasks.
- Training uses **BPTT**.
- Long sequences create **vanishing/exploding gradient** challenges.
- **LSTM** adds a gated cell state for stronger long-range memory.
- **GRU** is a simpler gated alternative.
- Practical implementations must handle batching, padding, masking, variable lengths and bidirectionality carefully.

## Week 4 Checklist

You should be able to answer:

1. Why are sequences different from unordered inputs?
2. What is an RNN hidden state?
3. Why are parameters shared across time?
4. What are many-to-one, one-to-many and many-to-many models?
5. Write the vanilla RNN update equation.
6. What is BPTT?
7. Why do gradients vanish or explode?
8. What is truncated BPTT?
9. What do the LSTM forget, input and output gates do?
10. How does GRU differ from LSTM?
11. When is bidirectional processing appropriate?
12. How should variable-length sequences be handled?